# 5. Classifying Labels as Dogs

**Rubric:**
- Matches between the classifier and the pet label are correctly classified as "dog" or "not dog"
- Non-matches between the classifier and the pet label are correctly classified as "dog" or "not dog"

A pet label and a classifier label can each independently be a dog or not, regardless of whether they *matched* each other exactly. `adjust_results4_isadog()` looks each one up in `dognames.txt` and records both answers -- this is what lets a breed *mismatch* (e.g. classifier said "walker hound" for an actual beagle) still correctly count as "yes, that's a dog".

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # so we can import the project's .py files


In [2]:
import inspect
from adjust_results4_isadog import adjust_results4_isadog
print(inspect.getsource(adjust_results4_isadog))

def adjust_results4_isadog(results_dic, dogfile):
    """
    Adds two more elements to each value in results_dic: whether the pet
    image's true label is a dog (1) or not (0), and whether the
    classifier's label is a dog (1) or not (0). A breed name only counts as
    "a dog" if it appears in dogfile.

    Parameters:
     results_dic - dictionary from classify_images(); each value list is
                   [pet_label, classifier_label, is_match] and this function
                   extends it in place to
                   [pet_label, classifier_label, is_match,
                    pet_label_is_dog, classifier_label_is_dog]
     dogfile - path to a text file listing valid dog breed names, one per
               line, lowercase

    Returns:
     None - results_dic is mutated in place
    """
    dognames_dic = dict()
    with open(dogfile, "r") as f:
        for line in f:
            name = line.strip().lower()
            if name and name not in dognames_dic:
                

## Running it and looking at all four combinations

In [3]:
from get_pet_labels import get_pet_labels
from classify_images import classify_images

images_dir = '../pet_images/'
results_dic = get_pet_labels(images_dir)
classify_images(images_dir, results_dic, 'alexnet')  # alexnet has the most edge cases
adjust_results4_isadog(results_dic, '../dognames.txt')

# Each value is now [pet_label, classifier_label, is_match, pet_is_dog, classifier_is_dog]
for filename, v in results_dic.items():
    pet_label, classifier_label, is_match, pet_is_dog, classifier_is_dog = v
    print(f'{filename:24s} match={is_match}  pet_is_dog={pet_is_dog}  classifier_is_dog={classifier_is_dog}')

Poodle_02.jpg            match=0  pet_is_dog=1  classifier_is_dog=1
Boxer_02.jpg             match=0  pet_is_dog=1  classifier_is_dog=1
Chihuahua_03.jpg         match=1  pet_is_dog=1  classifier_is_dog=1
Boxer_01.jpg             match=1  pet_is_dog=1  classifier_is_dog=1
Collie_02.jpg            match=1  pet_is_dog=1  classifier_is_dog=1
Saint_bernard_01.jpg     match=1  pet_is_dog=1  classifier_is_dog=1
Great_dane_02.jpg        match=1  pet_is_dog=1  classifier_is_dog=1
Poodle_03.jpg            match=1  pet_is_dog=1  classifier_is_dog=1
Golden_retriever_03.jpg  match=1  pet_is_dog=1  classifier_is_dog=1
Poodle_01.jpg            match=0  pet_is_dog=1  classifier_is_dog=1
Collie_01.jpg            match=0  pet_is_dog=1  classifier_is_dog=1
Pug_03.jpg               match=1  pet_is_dog=1  classifier_is_dog=1
German_shepherd_02.jpg   match=1  pet_is_dog=1  classifier_is_dog=1
Collie_03.jpg            match=1  pet_is_dog=1  classifier_is_dog=1
Daisy_01.jpg             match=1  pet_is_dog=0  

## The four cases the rubric is checking

1. **Match, both dogs** -- pet label and classifier label are the same breed, and both correctly register as dogs.
2. **Match, both not-dogs** -- e.g. a guitar photo correctly labeled "guitar" by the classifier -- both correctly register as not-a-dog.
3. **Non-match, but still both dogs** -- the classifier names the wrong *breed*, but that wrong breed is still a real dog breed, so both sides still register as "dog".
4. **Non-match, crossing the dog/not-dog line** -- the hardest case: a real dog photo the classifier mistakes for something that isn't a dog at all, or a non-dog photo the classifier mistakes for a dog breed.

The cells below pull one concrete example of each straight out of the results dictionary, so you can see exactly which image exercises which rubric case.

In [4]:
def show(label, predicate):
    print(f'--- {label} ---')
    for filename, v in results_dic.items():
        pet_label, classifier_label, is_match, pet_is_dog, classifier_is_dog = v
        if predicate(is_match, pet_is_dog, classifier_is_dog):
            print(f'  {filename:24s} pet={pet_label!r:18s} classifier={classifier_label!r:22s} '
                  f'pet_is_dog={pet_is_dog} classifier_is_dog={classifier_is_dog}')
            return
    print('  (no example found for this run)')

show('1. Match, both dogs', lambda m, p, c: m == 1 and p == 1 and c == 1)
show('2. Match, both not-dogs', lambda m, p, c: m == 1 and p == 0 and c == 0)
show('3. Non-match, still both dogs', lambda m, p, c: m == 0 and p == 1 and c == 1)
show('4a. Dog misclassified as NOT a dog', lambda m, p, c: p == 1 and c == 0)
show('4b. Not-a-dog misclassified as a dog', lambda m, p, c: p == 0 and c == 1)

--- 1. Match, both dogs ---
  Chihuahua_03.jpg         pet='chihuahua'        classifier='chihuahua'            pet_is_dog=1 classifier_is_dog=1
--- 2. Match, both not-dogs ---
  Daisy_01.jpg             pet='daisy'            classifier='daisy'                pet_is_dog=0 classifier_is_dog=0
--- 3. Non-match, still both dogs ---
  Poodle_02.jpg            pet='poodle'           classifier='afghan hound'         pet_is_dog=1 classifier_is_dog=1
--- 4a. Dog misclassified as NOT a dog ---
  Great_dane_03.jpg        pet='great dane'       classifier='wild boar'            pet_is_dog=1 classifier_is_dog=0
--- 4b. Not-a-dog misclassified as a dog ---
  Persian_cat_01.jpg       pet='persian cat'      classifier='toy poodle'           pet_is_dog=0 classifier_is_dog=1
